# Master Exam Template: Regression Models
## Course Scope: Simple & Multiple Linear Regression, Ridge, Lasso, and Random Forest
**Matches:** 7CS353 Lab Assignments 5, 6, 10  
**Marks Rubric (40 Marks):**
* Section 1: EDA [10 Marks]
* Section 2: Implementation (Linear Regression Baseline) [10 Marks]
* Section 3: Evaluation (MAE, MSE, RMSE, R², Residuals) [10 Marks]
* Section 4: Modify Technique (Ridge/Lasso Regularization & Random Forest with GridSearchCV) [10 Marks]

In [ ]:
# 1. Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams['figure.figsize'] = (8, 4)
print("Libraries imported!")

In [ ]:
# Section 1: EDA (10 Marks) - Quick Exam Dataset Generation
np.random.seed(42)
n = 400

x1 = np.random.uniform(10, 50, n)
x2 = np.random.uniform(1, 10, n)
x3 = np.random.randint(1, 6, n)
y = 12.0 + 2.5 * x1 + 4.0 * x2 - 1.5 * x3 + np.random.normal(0, 3, n)

df = pd.DataFrame({'Feature_1': x1, 'Feature_2': x2, 'Feature_3': x3, 'Target': y})

print("Dataset shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
display(df.describe().round(2))

# Correlation & Target Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df['Target'], kde=True, ax=axes[0], color='teal')
axes[0].set_title("Target Distribution")
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', ax=axes[1])
axes[1].set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
# Section 2: Implementation (10 Marks) - Train/Test Split & Baseline
X = df.drop(columns=['Target'])
y = df['Target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)
print("Baseline Linear Regression fitted!")

In [ ]:
# Section 3: Evaluation (10 Marks) - Metrics & Residuals
mae = mean_absolute_error(y_test, y_pred_lr)
mse = mean_squared_error(y_test, y_pred_lr)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_lr)

res_baseline = {'Model': 'Linear Regression (Baseline)', 'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'R2': round(r2, 4)}
display(pd.DataFrame([res_baseline]))

# Residual plot
plt.figure(figsize=(6, 3))
plt.scatter(y_pred_lr, y_test - y_pred_lr, color='purple', alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.title("Residuals vs Predicted")
plt.show()

In [ ]:
# Section 4: Modify Technique (10 Marks) - Ridge, Lasso & Tuned Random Forest
# 4.1 Ridge & Lasso
ridge = Ridge(alpha=1.0).fit(X_train_scaled, y_train)
lasso = Lasso(alpha=0.1).fit(X_train_scaled, y_train)

res_ridge = {'Model': 'Ridge (L2)', 'MAE': round(mean_absolute_error(y_test, ridge.predict(X_test_scaled)), 2), 'RMSE': round(np.sqrt(mean_squared_error(y_test, ridge.predict(X_test_scaled))), 2), 'R2': round(r2_score(y_test, ridge.predict(X_test_scaled)), 4)}
res_lasso = {'Model': 'Lasso (L1)', 'MAE': round(mean_absolute_error(y_test, lasso.predict(X_test_scaled)), 2), 'RMSE': round(np.sqrt(mean_squared_error(y_test, lasso.predict(X_test_scaled))), 2), 'R2': round(r2_score(y_test, lasso.predict(X_test_scaled)), 4)}

# 4.2 Tuned Random Forest (GridSearchCV)
grid_rf = GridSearchCV(RandomForestRegressor(random_state=42), {'n_estimators': [50, 100], 'max_depth': [3, 5]}, cv=5, scoring='r2')
grid_rf.fit(X_train, y_train)
best_rf = grid_rf.best_estimator_
res_rf = {'Model': f'Tuned Random Forest {grid_rf.best_params_}', 'MAE': round(mean_absolute_error(y_test, best_rf.predict(X_test)), 2), 'RMSE': round(np.sqrt(mean_squared_error(y_test, best_rf.predict(X_test))), 2), 'R2': round(r2_score(y_test, best_rf.predict(X_test)), 4)}

# Comparison Table
comparison_df = pd.DataFrame([res_baseline, res_ridge, res_lasso, res_rf])
print("=== Final Comparison Table ===")
display(comparison_df)